# Timing Test

In [1]:
import json
import statistics
import time
from collections import Counter
from pathlib import Path

import requests

BASE_URL = "http://127.0.0.1:8002"
SIZES = (10, 35, 100)
MODEL_FILES = [next(Path().glob(f"Timing_Test_-_{n}_Nodes_*.json")) for n in SIZES]
REQUEST_DIR = Path("requests")
WARMUP_RUNS = 5
RUNS = 30

In [2]:
bodies = {}
for path in MODEL_FILES:
    model = json.loads(path.read_text(encoding="utf-8"))
    body = (REQUEST_DIR / f"{path.stem}.request.json").read_bytes()
    request = json.loads(body)
    assert request["metadata"]["id"] == model["metadata"]["id"] and request["compilation_target"] == "qasm"
    name = model["metadata"]["name"]
    bodies[name] = body
    types = Counter(node["type"] for node in request["nodes"])
    print(f"{name}: {len(request['nodes'])} nodes, {len(request['edges'])} edges, node types: {dict(types.most_common())}")

Timing Test - 10 Nodes: 10 nodes, 9 edges, node types: {'gate': 7, 'encode': 1, 'measure': 1, 'int': 1}
Timing Test - 35 Nodes: 35 nodes, 35 edges, node types: {'gate': 20, 'encode': 5, 'measure': 5, 'int': 5}
Timing Test - 100 Nodes: 100 nodes, 100 edges, node types: {'gate': 65, 'measure': 15, 'encode': 5, 'int': 5, 'qubit': 5, 'prepare': 5}


In [3]:
session = requests.Session()


def compile_once(body: bytes) -> float:
    start = time.perf_counter()
    response = session.post(f"{BASE_URL}/debug/compile", data=body,
                            headers={"Content-Type": "application/json"}, timeout=600)
    elapsed = time.perf_counter() - start
    if response.status_code != 200 or not response.headers["content-type"].startswith("text/plain"):
        raise RuntimeError(f"compilation failed: {response.text[:500]}")
    return elapsed


print(f"{'model':<26}{'nodes':>6}{'median [ms]':>13}{'min [ms]':>10}{'max [ms]':>10}")
for name, body in bodies.items():
    nodes = len(json.loads(body)["nodes"])
    for _ in range(WARMUP_RUNS):
        compile_once(body)
    times = [1000 * compile_once(body) for _ in range(RUNS)]
    print(f"{name:<26}{nodes:>6}{statistics.median(times):>13.1f}{min(times):>10.1f}{max(times):>10.1f}")

model                      nodes  median [ms]  min [ms]  max [ms]


Timing Test - 10 Nodes        10          6.3       5.4       9.8


Timing Test - 35 Nodes        35         18.6      15.8      25.9


Timing Test - 100 Nodes      100         42.4      38.1     117.6
